# Working with Databases in Python
v.ekc

Databases let you store, query, and exchange data more powerfully than a CSV ever could. Today we'll use Python's built-in `sqlite3` library to create and query a database from scratch. Then we will learn how to move data back and forth between SQLite and pandas 🐼🐼🐼🐼🐼

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | Working with Databases — SQLite |
| 3 | Reading SQL Results into Pandas |
| 4 | Saving DataFrames to SQLite |
| 5 | 🔬 Open Exploration — Chinook Database |
| Appendix | Quick Reference |

---
## 1. Setup

In [1]:
import sqlite3
import pandas as pd
import numpy as np

---
## 2. Working with Databases: SQLite

Why use a database instead of a CSV?

- **Scale:** databases can handle data too large to fit in memory
- **Sharing:** multiple scripts or applications can query the same file
- **Querying:** SQL lets you retrieve exactly what you need without loading everything
- **Relationships:** multiple tables can be linked and joined together

SQLite is the simplest option....it's a single `.db` file, no server required, and Python ships with it built in. Perfect for learning and for local projects.



| Step | Code | Notes |
|------|------|-------|
| Connect | `sqlite3.connect('file.db')` | Creates the file if it doesn't exist |
| Cursor | `conn.cursor()` | Your tool for writing SQL commands |
| Execute | `cursor.execute("SQL")` | Run a single SQL statement |
| Execute many | `cursor.executemany("SQL", list)` | Insert multiple rows at once |
| Commit | `conn.commit()` | Save changes — don't skip this! |
| Close | `cursor.close(); conn.close()` | Always clean up when done |

### Connecting to a Database

`sqlite3.connect()` opens a connection to the database file. If the file doesn't exist yet, SQLite creates it automatically and you don't need to set anything up.

In [2]:
# Establish a connection — creates demo.db if it doesn't exist yet
dbconn = sqlite3.connect('demo.db')
type(dbconn)

sqlite3.Connection

In [3]:
# Confirm the database file now lives in your working directory
!ls *.db

chinook.db  demo.db     tutorial.db


The connection is your channel to the database. Now we need to create a `cursor` object. The cursor is a pointer to iterate/move through the database.

In [4]:
# Create a cursor to execute SQL statements
cursor = dbconn.cursor()
type(cursor)

sqlite3.Cursor

### Creating a Table

SQL uses a typed schema — you declare each column's name and data type upfront. SQLite supports: `TEXT`, `INTEGER`, `FLOAT`, `BLOB`, and `NULL`.

> **Don't forget to commit!** Changes aren't saved to disk until you call `conn.commit()`. Think of it like hitting save — if you skip it, your data vanishes when the connection closes.

In [6]:
# Create a books table with 4 typed columns
#    triple quotes help with readability of ur strings

cursor.execute("""
    CREATE TABLE books
    (title TEXT, author TEXT, price FLOAT, year INTEGER)
""")
dbconn.commit()

OperationalError: table books already exists

### Inserting Data

We can insert one row at a time with `cursor.execute()`, or hand off a whole list with `cursor.executemany()` for bulk inserts. Both need a `commit()` after.

In [8]:
# Insert rows one at a time
cursor.execute("""
    INSERT INTO books VALUES 
    ("Twilight", "Stephanie Meyer", 17.99, 2005)
""")
cursor.execute("""
    INSERT INTO books VALUES 
    ("A Court of Thorns and Roses", "Sarah J. Maas", 10.00, 2015)
""")
dbconn.commit()

In [ ]:
# Insert multiple rows in a single execute call
cursor.execute("""
    INSERT INTO books VALUES
        ("Pride and Prejudice", "Jane Austen", 12.99, 1813),
        ("To Kill a Mockingbird", "Harper Lee", 15.50, 1960)
""")
dbconn.commit()

When you have a Python list of tuples ready to go, `executemany()` is the cleanest approach — pass a parameterized query with `?` as placeholders, and SQLite fills them in from your list.

In [ ]:
# executemany — pass a list of tuples, use ? as value placeholders
books_list = [
    ("The Catcher in the Rye", "J.D. Salinger", 14.99, 1951),
    ("The Lord of the Rings", "J.R.R. Tolkien", 19.95, 1954),
    ("Moby-Dick", "Herman Melville", 11.75, 1851)
]

cursor.executemany("""
    INSERT INTO books VALUES (?,?,?,?)
""", books_list)
dbconn.commit()

Let's do a quick sanity check — `fetchall()` returns the query results as a **list of tuples**. Column names aren't included here (we'll fix that in Section 3).

In [ ]:
# Fetch all rows — returns a list of tuples
lstbooks = cursor.execute('SELECT * FROM books').fetchall()
lstbooks

In [ ]:
# Turn the list of tuples into a DataFrame
# (notice the column names are missing — we'll fix this in Section 3)
pd.DataFrame(lstbooks)

---
### Try It 1: Build Your Own Table 😊

Use the open connection (`dbconn` and `cursor`) to add a new table to `demo.db`.

1. Create a `movies` table with columns: `title` (TEXT), `director` (TEXT), `rating` (FLOAT), `year` (INTEGER).

2. Insert **at least 2 rows** one at a time using `cursor.execute()`.

3. Use `cursor.executemany()` to bulk-insert **at least 3 more rows** from a Python list.

4. Verify all your rows made it in: run `cursor.execute('SELECT * FROM movies').fetchall()`.

In [ ]:
# 1. Create movies table


In [ ]:
# 2. Insert 2 rows one at a time


In [ ]:
# 3. executemany — insert at least 3 more from a list


In [ ]:
# 4. Verify rows are there


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Use `?` as the placeholder in `executemany` — SQLite fills them in from your list of tuples, in order.*

```python
# 1.
cursor.execute("""
    CREATE TABLE movies
    (title TEXT, director TEXT, rating FLOAT, year INTEGER)
""")
dbconn.commit()

# 2.
cursor.execute('INSERT INTO movies VALUES ("Parasite", "Bong Joon-ho", 8.5, 2019)')
cursor.execute('INSERT INTO movies VALUES ("Spirited Away", "Hayao Miyazaki", 8.6, 2001)')
dbconn.commit()

# 3.
movies_list = [
    ("Inception", "Christopher Nolan", 8.8, 2010),
    ("Get Out", "Jordan Peele", 7.7, 2017),
    ("Everything Everywhere All at Once", "Daniel Kwan", 7.8, 2022)
]
cursor.executemany("INSERT INTO movies VALUES (?,?,?,?)", movies_list)
dbconn.commit()

# 4.
cursor.execute('SELECT * FROM movies').fetchall()
```

</details>

---
## 3. Reading SQL Results into Pandas

`pd.read_sql_query()` does two things at once: it runs your SQL *and* returns a DataFrame with proper column names. Much cleaner than calling `.fetchall()` and wrapping in `pd.DataFrame()` manually.

### 📋 Board Reference — SQL Query Patterns

| Goal | SQL |
|------|-----|
| All rows & columns | `SELECT * FROM table` |
| Specific columns | `SELECT col1, col2 FROM table` |
| Filter rows | `SELECT * FROM table WHERE condition` |
| Row by position | `SELECT * FROM table WHERE ROWID = n` |
| Limit + skip rows | `SELECT * FROM table LIMIT n OFFSET m` |
| Sort results | `SELECT * FROM table ORDER BY col DESC` |

In [ ]:
# pd.read_sql_query — runs SQL and returns a labeled DataFrame
dfbook = pd.read_sql_query("SELECT * FROM books", dbconn)
dfbook

> **Notice:** column names come through automatically — no need to reset them manually like we did with the raw `fetchall()` approach. 😊

In [ ]:
dfbook.dtypes

You can select specific columns by naming them in the `SELECT` clause. Only those columns come back — useful when you have a wide table and just need a few fields.

In [ ]:
# Select specific columns only
pd.read_sql_query('SELECT title, author, year FROM books', dbconn)

Use `WHERE` to filter rows by a condition. The filtering happens in SQL — before the data even touches Python — which is what makes databases fast for large datasets.

In [ ]:
# Filter rows: books priced under $10
pd.read_sql_query('SELECT * FROM books WHERE price < 10', dbconn)

SQLite automatically assigns each row a `ROWID` (1-indexed). You can use it to fetch a specific row by its position in the table.

In [ ]:
# Fetch a single row by position
pd.read_sql_query('SELECT * FROM books WHERE ROWID = 3', dbconn)

`LIMIT` caps how many rows come back. `OFFSET` skips the first *n* rows before counting — handy for pagination or sampling the middle of a large table.

In [ ]:
# Return 3 rows starting at row 3 (OFFSET 2 skips the first 2)
pd.read_sql_query('SELECT * FROM books LIMIT 3 OFFSET 2', dbconn)

SQL can handle more complex logic too. The query below uses a **window function** (`ROW_NUMBER()`) and the modulo operator to select every other row — the kind of thing that would take a few pandas steps.

In [ ]:
# Select every other row using ROW_NUMBER() and modulo
pd.read_sql_query('''SELECT * FROM (
               SELECT *, ROW_NUMBER() OVER () AS row_num FROM books
           ) AS numbered_rows
           WHERE row_num % 2 = 1''', dbconn)

---
### Try It 2: Writing SQL Queries 😊

Use `pd.read_sql_query()` and the open `dbconn` to answer these questions about the `books` table.

1. Select **only the `title` and `price` columns** from `books`.

2. Find all books published **before 1900**.

3. Find all books where `price` is **between $10 and $16** (inclusive). *Hint: use `AND` to combine two conditions.*

4. Return all books **sorted by price** from lowest to highest. *Hint: `ORDER BY col ASC`.*

In [ ]:
# 1. title and price only


In [ ]:
# 2. books published before 1900


In [ ]:
# 3. price between $10 and $16


In [ ]:
# 4. sorted by price ascending


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*SQL filtering happens server-side before pandas even sees the data — that's the real power of databases at scale.*

```python
# 1.
pd.read_sql_query('SELECT title, price FROM books', dbconn)

# 2.
pd.read_sql_query('SELECT * FROM books WHERE year < 1900', dbconn)

# 3.
pd.read_sql_query('SELECT * FROM books WHERE price >= 10 AND price <= 16', dbconn)

# 4.
pd.read_sql_query('SELECT * FROM books ORDER BY price ASC', dbconn)
```

</details>

---
## 4. Saving DataFrames to SQLite

What if you already have a cleaned-up pandas DataFrame and want to persist it in a database? `df.to_sql()` handles the entire schema creation and data insert in one line — no `CREATE TABLE` needed.

In [ ]:
# Load a DataFrame we want to save into the database
another_df = pd.read_csv('earthquakes.csv')
another_df.head()

`df.to_sql()` writes the DataFrame to a table in your database. The `if_exists` parameter controls what happens when the table already exists:

> - `'fail'` — raise an error (default, safe for production)
> - `'replace'` — drop and recreate the table (useful during development)
> - `'append'` — add rows to an existing table (for streaming new data in)

In [ ]:
# Save the DataFrame as a new table in demo.db
another_df.to_sql('earthquakes', con=dbconn, if_exists='replace')

The `sqlite_master` table is SQLite's internal directory — query it to see all tables currently in the database.

In [ ]:
# Check what tables exist in demo.db
pd.read_sql_query("SELECT * FROM sqlite_master WHERE type = 'table'", dbconn)

---
### Try It 3: DataFrame ↔ Database Round Trip 😊

Practice the full workflow in isolation — use a fresh database called `mydata.db` so you don't touch `demo.db`.

1. Open a connection to `mydata.db` and create a cursor.

2. Load **any CSV** you have on hand (or reuse `earthquakes.csv`) into a DataFrame.

3. Save it to `mydata.db` using `df.to_sql()` with `if_exists='replace'`.

4. Query the first **5 rows** of your table using `pd.read_sql_query()` with `LIMIT 5`.

5. Write a `WHERE` query to filter the data by at least one condition of your choice.

6. Drop your table, commit, and close the connection.

In [ ]:
# 1. Open connection to mydata.db
myconn = ...
mycur = ...

In [ ]:
# 2. Load a CSV
my_df = ...

In [ ]:
# 3. Save to database with to_sql


In [ ]:
# 4. Query first 5 rows


In [ ]:
# 5. WHERE query — filter by a condition of your choice


In [ ]:
# 6. Drop table, commit, close


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Opening a connection to a new filename is all it takes to create a fresh database — no other setup needed.*

```python
# 1.
myconn = sqlite3.connect('mydata.db')
mycur = myconn.cursor()

# 2.
my_df = pd.read_csv('earthquakes.csv')

# 3.
my_df.to_sql('earthquakes', con=myconn, if_exists='replace')

# 4.
pd.read_sql_query('SELECT * FROM earthquakes LIMIT 5', myconn)

# 5. (example — filter by magnitude)
pd.read_sql_query('SELECT * FROM earthquakes WHERE mag > 5.0', myconn)

# 6.
mycur.execute('DROP TABLE earthquakes')
myconn.commit()
mycur.close()
myconn.close()
```

</details>

### Cleaning Up

Use `DROP TABLE` to remove a table entirely. Always commit after. When you're completely done with a database session, close both the cursor and the connection to release the file lock.

In [ ]:
# Drop all tables from demo.db
# (IF EXISTS avoids an error if a Try It table wasn't created)
cursor.execute('DROP TABLE IF EXISTS movies')
cursor.execute('DROP TABLE books')
cursor.execute('DROP TABLE earthquakes')
dbconn.commit()

# Verify — should be empty now
pd.read_sql_query("SELECT * FROM sqlite_master WHERE type = 'table'", dbconn)

In [ ]:
# Close cursor and connection when you're done
cursor.close()
dbconn.close()

---
## 5. 🔬 Open Exploration — Chinook Database

The [Chinook database](http://www.sqlitetutorial.net/sqlite-sample-database/) is a sample music store database — think of it as a digital record shop with a full back-office. It has 11 tables covering tracks, albums, artists, customers, invoices, and more.

| Table | What's in it |
|-------|--------------|
| `tracks` | Songs: name, duration, price, genre |
| `albums` | Album titles linked to artists |
| `artists` | Artist names |
| `customers` | Customer info: name, country, email |
| `invoices` | Purchase records |
| `invoice_items` | Line items linking invoices to tracks |
| `genres` | Genre names |
| `playlists` | Playlist names |
| `playlist_track` | Many-to-many: playlists ↔ tracks |
| `employees` | Employee info |
| `media_types` | Audio format info |

### ETL

Before we explore, let's get oriented. What tables do we have? What does the schema look like?

In [ ]:
# Confirm chinook.db is in your working directory, then connect
!ls *.db

conn = sqlite3.connect('chinook.db')
cur = conn.cursor()

In [ ]:
# What tables are in the Chinook database?
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)

In [ ]:
# Peek at the tracks table
tracks_df = pd.read_sql_query('SELECT * FROM tracks', conn)
tracks_df.head()

In [ ]:
# How many tracks? What are the column types?
tracks_df.info()

In [ ]:
# Peek at the artists table
artists_df = pd.read_sql_query('SELECT * FROM artists', conn)
artists_df.head()

---
### Try It 4: Explore the Chinook Database 😊

Use the open `conn` to dig into the data.

1. Read the `albums` table into a DataFrame called `albums_df` and preview it.

2. Merge `tracks_df` and `albums_df` (inner join). How many rows does the merged DataFrame have? What does that tell you?

3. Read the `customers` table. How many **unique countries** are represented? Use pandas `.nunique()` to find out.

4. Write a **SQL `WHERE` query** (not pandas) to find all tracks with a `UnitPrice` greater than `0.99`. How many rows come back?

5. **Your choice:** pick any two tables from the table above, read them both in, and merge them. What shared key did you use? What does the merged table represent?

In [ ]:
# 1. Read albums table
albums_df = ...
albums_df.head()

In [ ]:
# 2. Merge tracks_df and albums_df (inner join)


In [ ]:
# 3. Read customers table, count unique countries


In [ ]:
# 4. SQL WHERE: tracks with UnitPrice > 0.99


In [ ]:
# 5. Your choice: pick two tables and merge them


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*The `tracks` and `albums` tables share `AlbumId` — a standard inner merge picks it up automatically. Each track appears once, paired with its album title.*

```python
# 1.
albums_df = pd.read_sql_query('SELECT * FROM albums', conn)

# 2.
tracks_df.merge(albums_df)   # inner join on AlbumId

# 3.
customers_df = pd.read_sql_query('SELECT * FROM customers', conn)
customers_df['Country'].nunique()

# 4.
pd.read_sql_query('SELECT * FROM tracks WHERE UnitPrice > 0.99', conn)

# 5. Example — join genres onto tracks
genres_df = pd.read_sql_query('SELECT * FROM genres', conn)
tracks_df.merge(genres_df, on='GenreId')
```

</details>

---
### Open Exploration 🔬

Now go deeper — use a mix of SQL queries and pandas to answer questions you actually find interesting.

**Prompt 1 — Counts & Distributions:** How many tracks are in each genre? Use `pd.read_sql_query()` to get genre *names* alongside the counts. *Hint: you'll need to join `tracks` and `genres` on `GenreId`, then use pandas `.value_counts()` or a SQL `GROUP BY`.*

In [ ]:
# Prompt 1 — tracks per genre


<details>
<summary>💡 One approach — click to peek</summary>
<br>

*Merging first, then using pandas `.value_counts()` is the most readable approach here.*

```python
genres_df = pd.read_sql_query('SELECT * FROM genres', conn)
tracks_df.merge(genres_df, on='GenreId')['Name_y'].value_counts()

# or with SQL GROUP BY:
pd.read_sql_query('''
    SELECT genres.Name, COUNT(tracks.TrackId) AS track_count
    FROM tracks
    JOIN genres ON tracks.GenreId = genres.GenreId
    GROUP BY genres.Name
    ORDER BY track_count DESC
''', conn)
```

</details>

**Prompt 2 — Customer Geography:** Which countries have the most customers? Use pandas to find the top 10 countries and display the result. What does this tell you about where the store's customer base is concentrated?

In [ ]:
# Prompt 2 — top 10 customer countries


<details>
<summary>💡 One approach — click to peek</summary>
<br>

```python
customers_df = pd.read_sql_query('SELECT * FROM customers', conn)
customers_df['Country'].value_counts().head(10)
```

</details>

**Prompt 3 — WiLdCaRd:** Ask a question *you* find interesting about the Chinook data. Use SQL, pandas, or both to answer it. Requirements:
- Use **at least 2 tables**
- Apply at least **one filter** (SQL `WHERE` or pandas boolean indexing)

Write one sentence below interpreting your result.

In [ ]:
# Prompt 3 — your question


*Your interpretation: (double-click to edit)*

<details>
<summary>💡 One example — click to peek</summary>
<br>

*Many right answers here! This example finds the top-selling artists by total invoice revenue.*

```python
# Which artists generated the most revenue?
invoice_items_df = pd.read_sql_query('SELECT * FROM invoice_items', conn)
(
    tracks_df
    .merge(albums_df, on='AlbumId')
    .merge(artists_df, on='ArtistId')
    .merge(invoice_items_df, on='TrackId')
    .groupby('Name')['UnitPrice_y']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
```

</details>

In [ ]:
# Close the Chinook connection when done
cur.close()
conn.close()

---
## Appendix — SQLite + Pandas Quick Reference

### Minimal template
```python
import sqlite3
import pandas as pd

conn = sqlite3.connect('mydb.db')
cursor = conn.cursor()

cursor.execute("CREATE TABLE t (col1 TEXT, col2 INT)")
cursor.execute("INSERT INTO t VALUES ('a', 1)")
conn.commit()

df = pd.read_sql_query("SELECT * FROM t", conn)

cursor.close()
conn.close()
```

### Full template
```python
import sqlite3
import pandas as pd

# Connect
conn = sqlite3.connect('mydb.db')
cursor = conn.cursor()

# Create table
cursor.execute("CREATE TABLE t (col1 TEXT, col2 INT)")
conn.commit()

# Insert one row
cursor.execute("INSERT INTO t VALUES ('a', 1)")
conn.commit()

# Bulk insert
cursor.executemany("INSERT INTO t VALUES (?,?)", [('b', 2), ('c', 3)])
conn.commit()

# Query into pandas
pd.read_sql_query("SELECT * FROM t", conn)
pd.read_sql_query("SELECT col1 FROM t WHERE col2 > 1", conn)
pd.read_sql_query("SELECT * FROM t ORDER BY col2 DESC LIMIT 5", conn)

# DataFrame → database
df.to_sql('table_name', con=conn, if_exists='replace')
# if_exists options: 'fail' | 'replace' | 'append'

# Check all tables
pd.read_sql_query("SELECT * FROM sqlite_master WHERE type='table'", conn)

# Drop a table
cursor.execute("DROP TABLE IF EXISTS t")
conn.commit()

# Clean up
cursor.close()
conn.close()
```